In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd
import time
import random
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
EDA_TABLES_DIR = PROCESSED_DIR / "eda_tables"

In [1]:
cities = [
    'Amsterdam', 'Athens', 'Barcelona', 'Berlin', 'Budapest',
    'Lisbon', 'London', 'Paris', 'Rome', 'Vienna',
    # 5 additional cities
    'Madrid', 'Milan', 'Prague', 'Dublin', 'Copenhagen'
]
cities

['Amsterdam',
 'Athens',
 'Barcelona',
 'Berlin',
 'Budapest',
 'Lisbon',
 'London',
 'Paris',
 'Rome',
 'Vienna',
 'Madrid',
 'Milan',
 'Prague',
 'Dublin',
 'Copenhagen']

In [3]:
import time
import random
import re
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

# --- ROBUST RATING/FAV EXTRACTOR ---
def get_details_and_fav(driver, wait):
    rating, reviews, is_fav = "N/A", "0", 0 

    try:
        # Locate the review anchor (link or button)
        try:
            review_link = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//a[contains(@href, 'reviews')] | //button[contains(., 'reviews')]")
            ))
        except:
            # Fallback for "New" listings
            try:
                h1_parent = driver.find_element(By.TAG_NAME, "h1").find_element(By.XPATH, "./..").text
                if "New" in h1_parent: return "New", "0", 0
            except: pass
            return rating, reviews, is_fav

        # Get text content (cleanly)
        link_text = review_link.get_attribute("textContent").strip()
        clean_link = " ".join(link_text.split())
        
        # Get parent text (for standard listings where rating is outside link)
        parent_text = review_link.find_element(By.XPATH, "./..").get_attribute("textContent").strip()
        clean_parent = " ".join(parent_text.split())

        # --- SCENARIO A: GUEST FAVORITE ---
        if "Guest favorite" in clean_link or "Guest favorite" in clean_parent:
            is_fav = 1
            # Review logic for fav: read the link text first (the count sits inside the link)
            rev_match = re.search(r"(\d+)\s*review", clean_link, re.IGNORECASE)
            if not rev_match:
                rev_match = re.search(r"\d\.\d{2}\s*(\d+)\s*review", clean_parent, re.IGNORECASE)
            if rev_match: reviews = rev_match.group(1)
            # Rating logic for fav: take the whole decimal, then remove review digits stuck to it
            r_match = re.search(r"(\d\.\d+)", clean_parent)
            if r_match:
                rating = r_match.group(1)
                if len(rating.split(".")[1]) > 2 and reviews != "0" and rating.endswith(reviews):
                    rating = rating[:-len(reviews)]
            # Most reliable: read each small text piece separately so digits never get glued together
            try:
                parent_el = review_link.find_element(By.XPATH, "./..")
                pieces = driver.execute_script(
                    "return Array.from(arguments[0].querySelectorAll('*'))"
                    ".filter(n => n.children.length === 0)"
                    ".map(n => n.textContent.trim()).filter(Boolean);", parent_el)
                got_rating, got_reviews = None, None
                for t in pieces:
                    if got_rating is None and re.fullmatch(r"\d\.\d{1,2}", t):
                        got_rating = t
                    m = re.fullmatch(r"(\d+)\s*reviews?", t, re.IGNORECASE)
                    if got_reviews is None and m:
                        got_reviews = m.group(1)
                if got_rating: rating = got_rating
                if got_reviews: reviews = got_reviews
            except: pass

        # --- SCENARIO B: STANDARD ---
        else:
            is_fav = 0
            if "New" in clean_parent:
                rating = "New"
            else:
                r_match = re.search(r"(\d\.\d{1,2})", clean_parent)
                if r_match: rating = r_match.group(1)

            # Try link first, then parent for reviews
            rev_match = re.search(r"(\d+)\s*review", clean_link, re.IGNORECASE)
            if not rev_match:
                rev_match = re.search(r"(\d+)\s*review", clean_parent, re.IGNORECASE)
            if rev_match: reviews = rev_match.group(1)

    except: pass
    return rating, reviews, is_fav
# --- PRICE EXTRACTOR (reads the booking box text instead of a long XPath) ---
def get_price(driver):
    pattern = r"^[ \t]*(\S[^\n]*\d[^\n]*?)[ \t]+for[ \t]+\d+[ \t]+nights[ \t]*$"
    def _find(d):
        try:
            txt = d.find_element(By.XPATH, "//div[@data-plugin-in-point-id='BOOK_IT_SIDEBAR']").text
        except:
            return False
        m = re.search(pattern, txt, re.MULTILINE)
        return m.group(1).strip() if m else False
    return WebDriverWait(driver, 10).until(_find)

RAW_DIR.mkdir(parents=True, exist_ok=True)
all_data = []
city_count = 0

for city in cities:
    city_count += 1
    
    # --- MAIN SETUP ---
    URL = f"https://www.airbnb.com/s/{city}/homes?refinement_paths%5B%5D=%2Fhomes&date_picker_type=calendar&search_type=filter_change&flexible_trip_lengths%5B%5D=one_week&monthly_start_date=2026-02-01&monthly_length=3&monthly_end_date=2026-05-01&price_filter_input_type=2&channel=EXPLORE&price_filter_num_nights=5&min_bedrooms=2&selected_filter_order%5B%5D=min_bedrooms%3A2&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt&update_selected_filters=true&query={city}&search_mode=regular_search&room_types%5B%5D=Entire%20home%2Fapt"

    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--start-maximized")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    target_count = 50  # Target number of items to scrape per city
    # Initialize a local counter
    items_scraped_this_city = 0 
    try:
        driver.get(URL)
        wait = WebDriverWait(driver, 8)
        time.sleep(3)
        try:
            driver.find_element(By.XPATH, "//button[normalize-space()='Got it']").click()
        except: pass

        while items_scraped_this_city < target_count:
            print(f"\n=== SCRAPING CITY {city_count}/{len(cities)}: {city} ===\n")
            print(f"--- Scraping Page (Collected: {len(all_data)}) ---")
            
            try:
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[data-testid='card-container']")))
            except: break

            listings = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']")
            listing_urls = []
            for l in listings:
                try:
                    url = l.find_element(By.TAG_NAME, "a").get_attribute("href")
                    listing_urls.append(url)
                except: continue

            main_window = driver.current_window_handle

            for url in listing_urls:
                if items_scraped_this_city >= target_count: break
                
                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
                    
                    # A. Name
                    try: name = driver.find_element(By.TAG_NAME, "h1").text
                    except: name = "No Title"

                    # B. Superhost
                    is_superhost = "No"
                    try:
                        driver.execute_script("window.scrollBy(0, 500);")
                        time.sleep(0.5) 
                        body_text = driver.find_element(By.TAG_NAME, "body").text
                        if "Superhost" in body_text: is_superhost = "Yes"
                    except: pass

                    # C. Price (Robust)
                    price = get_price(driver)
                    
                    # D. Rating/Fav (Robust)
                    rating, reviews, is_fav = get_details_and_fav(driver, wait)

                    all_data.append({
                        "Name": name, "city": city, "Price": price, "Superhost": is_superhost,
                        "is_fav": is_fav, "Rating": rating, "Reviews": reviews, "URL": url
                    })

                    print(f"[{len(all_data)}] {price} |{city}| ★{rating} ({reviews}) | Fav: {is_fav} | SH: {is_superhost}|")
                    time.sleep(random.uniform(1.0, 2.0))

                    driver.close()
                    driver.switch_to.window(main_window)
                    
                    # Increment the counter for successfully scraped items
                    items_scraped_this_city += 1
                    
                except Exception:
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)
                    continue

            if items_scraped_this_city < target_count:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", next_btn)
                    time.sleep(4)
                except: break

    finally:
        print(f"\nCompleted! Total scraped: {len(all_data)}")
        driver.quit()
        pd.DataFrame(all_data).to_csv(RAW_DIR / "Airbnb_Scraped_Backup.csv", index=False)

    if items_scraped_this_city == 0:
        print(f"!! Got 0 listings for {city}. Probably blocked, stopping to avoid a longer block.")
        break
    time.sleep(random.uniform(20, 40))


=== SCRAPING CITY 1/15: Amsterdam ===

--- Scraping Page (Collected: 0) ---
[1] ج.م 51,630 |Amsterdam| ★4.78 (819) | Fav: 0 | SH: Yes|
[2] ج.م 65,466 |Amsterdam| ★4.97 (322) | Fav: 1 | SH: Yes|
[3] ج.م 101,624 |Amsterdam| ★4.92 (686) | Fav: 1 | SH: Yes|
[4] ج.م 67,717 |Amsterdam| ★4.7 (325) | Fav: 0 | SH: No|
[5] ج.م 187,948 |Amsterdam| ★4.85 (763) | Fav: 1 | SH: Yes|
[6] ج.م 150,280 |Amsterdam| ★4.81 (771) | Fav: 0 | SH: No|
[7] ج.م 120,842 |Amsterdam| ★4.67 (201) | Fav: 0 | SH: No|
[8] ج.م 94,833 |Amsterdam| ★N/A (0) | Fav: 0 | SH: No|
[9] ج.م 77,696 |Amsterdam| ★4.35 (26) | Fav: 0 | SH: No|
[10] ج.م 185,934 |Amsterdam| ★New (1) | Fav: 0 | SH: Yes|
[11] ج.م 120,040 |Amsterdam| ★4.86 (138) | Fav: 1 | SH: Yes|
[12] ج.م 68,341 |Amsterdam| ★4.94 (117) | Fav: 1 | SH: Yes|

=== SCRAPING CITY 1/15: Amsterdam ===

--- Scraping Page (Collected: 12) ---
[13] ج.م 145,309 |Amsterdam| ★N/A (0) | Fav: 0 | SH: Yes|
[14] ج.م 333,811 |Amsterdam| ★4.92 (13) | Fav: 0 | SH: No|
[15] ج.م 146,329 |Amster

In [4]:
df_scraped = pd.DataFrame(all_data)
df_scraped

,Name,city,Price,Superhost,is_fav,Rating,Reviews,URL
0,"2-Bedroom with Bath, Sleeps 6, Amstelveen, 75m2",Amsterdam,"ج.م 51,630",Yes,0,4.78,819,https://www.airbnb.com/rooms/33092943?search_m...
1,Cozy houseboat with parking in the center of A...,Amsterdam,"ج.م 65,466",Yes,1,4.97,322,https://www.airbnb.com/rooms/58087008775012407...
2,Cozy Studios in Heart of Pijp,Amsterdam,"ج.م 101,624",Yes,1,4.92,686,https://www.airbnb.com/rooms/9323377?search_mo...
3,"Numa | 1 Bedroom Apartment w/ Sofa Bed, Double...",Amsterdam,"ج.م 67,717",No,0,4.7,325,https://www.airbnb.com/rooms/11454693974405935...
4,Spacious Serviced Apartment with River View,Amsterdam,"ج.م 187,948",Yes,1,4.85,763,https://www.airbnb.com/rooms/42446371?search_m...
...,...,...,...,...,...,...,...,...
745,Cozy loft apartment with best view to Kings Ga...,Copenhagen,"ج.م 163,200",Yes,0,4.81,16,https://www.airbnb.com/rooms/16592758016540117...
746,Prime Location - Near Nyhavn & Tivoli Gardens,Copenhagen,"ج.م 101,171",Yes,1,4.93,45,https://www.airbnb.com/rooms/14021333787645005...
747,Spacious Residence in Central CPH | Sleeps 4,Copenhagen,"ج.م 157,568",Yes,1,5.0,25,https://www.airbnb.com/rooms/16884077465524113...
748,Stay Central - Steps from Tivoli & Nyhavn,Copenhagen,"ج.م 109,506",Yes,1,4.97,61,https://www.airbnb.com/rooms/14021363767238027...


In [5]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
df_scraped.to_csv(RAW_DIR / "Airbnb_Scraped_Data.csv", index=False)